# 01: Auto Loader Basics

**Exam objective (from Data Ingestion and Loading domain):** Use Auto Loader 
with schema enforcement and schema evolution in batch modes (for example, 
directory listing or file notification) to land data into Unity-Catalog–
governed tables.

**Free Edition note:** Source files are written to a UC volume rather than 
to external cloud storage. The Auto Loader syntax (`cloudFiles` source) is 
identical regardless of storage backend; only the path prefix differs.

In [0]:
%sql
USE CATALOG certprep;
USE SCHEMA ingestion;

-- Create a managed volume to hold source files
CREATE VOLUME IF NOT EXISTS certprep.ingestion.raw_files;

## Volumes
A volume is a Unity Catalog securable that represents non-tabular file storage, or in other words, a governed place to put files.

Volumes are the third major UC data object alongside tables and views. Volumes live inside schemas just like a table or view and inherit access controls of the catalog/schema hierarchy.

### Managed vs External Volumes
Managed vs. external volumes mirror the managed-vs-external distinction for tables. Managed volumes live in UC-managed storage; external volumes point at storage you control. You created a managed one with CREATE VOLUME (no LOCATION clause).

### Why Volumes?
Before volumes, you had two bad options for file-based work in UC: store files in DBFS (not governed) or work with cloud paths directly (also not governed). Volumes give you governed file storage with the same access-control mechanisms you used for tables in the governance exercises.

In [0]:
import json
import os
from datetime import datetime, timedelta

volume_path = "/Volumes/certprep/ingestion/raw_files"

# Make sure the directory is empty before starting
dbutils.fs.rm(volume_path, recurse=True)
dbutils.fs.mkdirs(volume_path)

# Generate three small JSON files representing daily sales batches
for day_offset, batch in enumerate([
    [
        {"sale_id": 1, "region": "North", "amount": 1250.00, "sale_date": "2026-01-15"},
        {"sale_id": 2, "region": "South", "amount": 890.50,  "sale_date": "2026-01-15"},
    ],
    [
        {"sale_id": 3, "region": "East",  "amount": 2100.75, "sale_date": "2026-01-16"},
        {"sale_id": 4, "region": "West",  "amount": 1575.25, "sale_date": "2026-01-16"},
    ],
    [
        {"sale_id": 5, "region": "North", "amount": 3200.00, "sale_date": "2026-01-17"},
        {"sale_id": 6, "region": "South", "amount": 450.25,  "sale_date": "2026-01-17"},
    ],
]):
    file_path = f"{volume_path}/sales_2026-01-{15 + day_offset:02d}.json"
    # Write as newline-delimited JSON
    content = "\n".join(json.dumps(r) for r in batch)
    dbutils.fs.put(file_path, content, overwrite=True)

# Verify
display(dbutils.fs.ls(volume_path))

The above cell is merely to simulate files being dropped into the ingestion source location.

In [0]:
checkpoint_path = "/Volumes/certprep/ingestion/raw_files/_checkpoints/sales_autoloader"
target_table = "certprep.ingestion.sales_raw"

# Make sure checkpoint is fresh for this exercise
dbutils.fs.rm(checkpoint_path, recurse=True)

# Read with Auto Loader, write to a Delta table
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(volume_path)
.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

## Auto Loader

- `.format("cloudFiles")` is what activates the Auto Loader itself. Without it, `readStream` runs normally but for every file in the volume every single run. `cloudFiles` adds the file-tracking behavior on top of the streaming read.
- `.option("cloudFiles.format", "json")` specifies the format of the source files for the Auto Loader. Options include json, csv, parquet, avro, orc, binaryFile, text. 
    - `cloudFiles.format` controls parsing, not filtering. Files in the source directory that don't match the expected format will be picked up and attempted to be parsed, producing malformed-record errors or corrupt data. To filter by extension, use `pathGlobFilter`.
- `.option("cloudFiles.schemaLocation", checkpoint_path)` sets the path for AutoLoader to store its inferred schema. First run, Auto Loader infers the schema from the data and saves it here. On subsequent runs, Auto Loader loads it from here. This enables the read-side schema evolution behavior when the source schema changes.
- `.trigger(availableNow=True)` : Spark Structured Streaming offers three trigger modes: 
    - `processingTime="X seconds"`: continuous streaming, process micro-batches every X seconds, run forever.
    - `once=True`: process all currently available data in a single micro-batch, then stop (deprecated)
    - `availableNow=True`: process all currently available data, possibly across multiple micro-batches if the data is large, then stop
- `availableNow` is the modern replacement for `once`. Works for a scheduled batch-load ingestion job. The most common Auto Loader pattern in prod. 
- `checkpointLocation` on the write side: Spark Structured Streaming uses checkpoints to track progress across runs. The checkpoint stores:
    - offsets - what's been read from the source
    - commits - what's been written
    - source-specific state - Auto Loader's file tracking lives here
- Without a checkpoint, the stream has no memory and would reprocess everything every run

A quirk worth knowing: in our exercise, `schemaLocation` and `checkpointLocation` are the same path. Auto Loader allows this as a convenience — when both are the same, Auto Loader puts its schema info in a `_schemas` subdirectory within the checkpoint. In practice this is fine and common. The exam may ask whether they can be the same path (yes).


In [0]:
display(spark.table(target_table))

In [0]:
# Add a new day's file
new_batch = [
    {"sale_id": 7, "region": "East",  "amount": 1800.00, "sale_date": "2026-01-18"},
    {"sale_id": 8, "region": "West",  "amount": 2250.50, "sale_date": "2026-01-18"},
]
dbutils.fs.put(
    f"{volume_path}/sales_2026-01-18.json",
    "\n".join(json.dumps(r) for r in new_batch),
    overwrite=True,
)

# Now re-run the same Auto Loader read
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(volume_path)
.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

## Incremental Behavior
Upon adding a fourth file, the checkpoint from the first run recorded that the first three files were processed. On the second run, Auto Loader listed the directory, compared to the checkpoint, and found the new file. It processed only that file and updated the checkpoint to reflect such.

**Key Point** -> Checkpoints record what files have been processed. Each run is the difference between what is in the directory now vs what does the checkpoint list as already processed. 

Deleting a checkpoint will result in Auto Loader processing everything on the next run. 

Deleting or moving source files after processing completes does not impact Auto Loader on the next run. 

What happens if you change a file in-place (same name, different content): depends on the cloud provider and configuration. Default behavior is that Auto Loader keys off the file path, so re-writes are not detected. There are options `(cloudFiles.allowOverwrites, file-version-tracking)` to change this behavior, but they're not default and probably not exam-tested at the associate level.

In [0]:
display(spark.table(target_table).orderBy("sale_id"))

In [0]:
display(dbutils.fs.ls(checkpoint_path))

In [0]:
# Add a file with a new column not seen before
evolved_batch = [
    {"sale_id": 9, "region": "North", "amount": 500.00, "sale_date": "2026-01-19", "customer_tier": "gold"},
    {"sale_id": 10, "region": "South", "amount": 750.00, "sale_date": "2026-01-19", "customer_tier": "silver"},
]
dbutils.fs.put(
    f"{volume_path}/sales_2026-01-19.json",
    "\n".join(json.dumps(r) for r in evolved_batch),
    overwrite=True,
)

In [0]:
try:
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load(volume_path)
    .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )
    print("Run completed successfully")
except Exception as e:
    print(f"Run failed: {type(e).__name__}: {e}")

In [0]:
display(spark.table(target_table).orderBy("sale_id"))

1. Read-side evolution (Auto Loader's understanding of source data): controlled by `cloudFiles.schemaEvolutionMode`. Default is addNewColumns. When new columns appear in source data, Auto Loader's behavior depends on this setting:
    - `addNewColumns` (default): stop the stream, evolve the schema, restart. The trade-off is that new columns cause a stream restart, which means a brief interruption — fine for batch-style jobs, more disruptive for always-on streaming.
    - `rescue`: ignore the new column in the schema, dump its value into a _rescued_data JSON column.
    - `failOnNewColumns`: stop the stream and refuse to continue.
    - `none`: don't infer or evolve schema; you provide it explicitly.
2. Write-side evolution (Delta table's willingness to accept new columns): controlled by `mergeSchema` option on the write. By default, Delta tables reject writes whose schema doesn't match the table's current schema. `mergeSchema=true` tells Delta to add the new columns to the table schema instead of rejecting.

Why both are needed. Auto Loader evolving its read schema means "I now understand there's a `customer_tier` column in the incoming data." But that doesn't tell the target Delta table to make room for it. The Delta write fails with `DELTA_METADATA_MISMATCH` because the table schema doesn't have `customer_tier`, and the write isn't allowed to add it without `mergeSchema=true`.

Why this design. Spark Structured Streaming traditionally requires the query's output schema to be stable for the lifetime of the streaming query, because downstream operations (joins, aggregations, sinks) depend on it. Auto Loader's "stop and restart" behavior for `addNewColumns` is how it works around this — by stopping the query, evolving the schema, and starting a new query with the new schema. The Delta side has its own evolution mechanism (`mergeSchema`) that exists for similar correctness reasons.



## Schema Evolution Modes for Auto Loader

Auto Loader's read-side schema evolution is controlled by 
`cloudFiles.schemaEvolutionMode`. Four modes exist:

**`addNewColumns` (default)**
- Behavior: When a new column appears in source data, the stream stops, 
  Auto Loader updates its schema to include the new column, and the next 
  run picks up with the new schema. Existing rows in the target table 
  have NULL for the new column.
- Requires: `mergeSchema=true` on the Delta write for end-to-end evolution.
- Trade-off: New columns cause a stream restart, which means a brief 
  interruption — fine for batch-style jobs, more disruptive for 
  always-on streaming.

**`rescue`**
- Behavior: The schema never changes. Any column not in the known schema 
  is captured into a `_rescued_data` column as a JSON string. Existing 
  columns are read normally.
- Use case: You want to keep ingesting without any schema disruption, 
  and you'll handle unexpected fields downstream (e.g., a separate job 
  parses `_rescued_data` for analysis or alerting).

**`failOnNewColumns`**
- Behavior: Hard failure when a new column is detected. The stream stops 
  and refuses to continue until either the schema is manually updated 
  or the offending data is removed.
- Use case: Strict pipelines where unexpected schema changes indicate 
  an upstream bug or contract violation that must be reviewed by a human 
  before the pipeline proceeds.

**`none`**
- Behavior: No schema inference or evolution. The schema is whatever you 
  provide explicitly via `.schema(...)`. New columns in source data are 
  silently ignored.
- Use case: You have a strictly defined contract with the upstream system 
  and want to enforce that contract by ignoring anything outside it. 
  Common in regulated environments where adding fields shouldn't 
  silently propagate.

**When to use which mode:**

- `addNewColumns`: Default for most batch ingestions. Upstream may add 
  optional fields and you want them to flow through without manual 
  intervention. Combined with `mergeSchema=true`, end-to-end evolution 
  is automatic.

- `rescue`: When the target table's schema is owned by someone else 
  (e.g., a downstream consumer team) and you can't unilaterally evolve 
  it, but you still need to capture the data. Lets you preserve 
  unexpected fields without changing the contract.

- `failOnNewColumns`: When schema changes require formal review — 
  regulated data, contract-bound pipelines, or production systems where 
  silent evolution would create downstream surprises.

- `none`: When the upstream contract is fixed and you want hard 
  enforcement that nothing outside the agreed schema enters the pipeline.


## File Discovery Modes for Auto Loader

Auto Loader has to answer one question on every run: "what files have 
appeared in the source directory since I last checked?" Two strategies 
exist.

**Directory listing mode (default)**

How it works: Auto Loader lists the source directory and compares 
the listing to its checkpoint. Files not yet seen are processed.

Cost characteristic: scales with the *total number of files* in the 
directory. Every run lists everything, even if only one new file has 
arrived. For a directory with 50 million files, every run pays the 
cost of listing all 50 million.

Setup: zero — works out of the box. The default mode.

When appropriate: small-to-medium directories, development, simple 
pipelines, or any case where the listing cost isn't a problem.

**File notification mode**

How it works: Auto Loader subscribes to a cloud-provider notification 
service. The cloud provider emits an event whenever a new file is 
created in the bucket; Auto Loader's queue consumer receives the 
event and processes the new file. No directory listing required.

Cloud-specific infrastructure required:
- AWS: SNS topic + SQS queue subscribed to the bucket's events
- Azure: Event Grid + Storage Queue subscribed to the storage account
- GCP: Pub/Sub topic subscribed to the bucket's notifications

Cost characteristic: scales with the *rate of new files*, not the total 
file count. A directory with 50 million existing files and 1000 new 
files per day costs no more than a directory with 1000 files total.

Setup: requires configuring the notification infrastructure in the 
cloud provider, granting Auto Loader permission to read from the queue, 
and enabling the mode via `cloudFiles.useNotifications=true` (and 
related options).

When appropriate: high-volume production pipelines where the listing 
cost becomes significant. Databricks recommends file notification for 
pipelines with millions of files in the source directory.

**Key exam-relevant points:**

1. Directory listing is the default; file notification is opt-in.
2. The choice is driven by total file count, not by file size or 
   ingestion frequency.
3. File notification requires cloud-side infrastructure setup that 
   Auto Loader cannot create itself — it must be configured separately.
4. Both modes use the same Auto Loader API; only options differ.

In [0]:
# Drop the target table
spark.sql("DROP TABLE IF EXISTS certprep.ingestion.sales_raw")

In [0]:
# Drop the volume — this removes all files inside it, including source files and checkpoint
spark.sql("DROP VOLUME IF EXISTS certprep.ingestion.raw_files")

In [0]:
# Verify cleanup
display(spark.sql("SHOW TABLES IN certprep.ingestion"))
display(spark.sql("SHOW VOLUMES IN certprep.ingestion"))

## Self Check Questions

1. The `cloudFiles.schemaLocation` and the `checkpointLocation` look 
   similar — both are paths Auto Loader writes to. What's the difference 
   between what each one stores? Can they be the same path?

2. You delete the checkpoint directory and re-run an Auto Loader read 
   against a source directory that still has all its files. What happens? 
   Why?

3. A new file with a new column arrives. The schema evolution mode is 
   `addNewColumns` (default). On the first run after the new file arrives, 
   what does Auto Loader do? On the second run? Why does it take two runs?

4. Your team is ingesting from a directory that contains 50 million files, 
   growing by ~10,000 new files per day. Which discovery mode is appropriate, 
   and why?

5. Auto Loader is reading JSON files (`cloudFiles.format = json`). A CSV file 
   is accidentally dropped into the same source directory. What happens? 
   Reason about this — don't assume.

6. You're using Auto Loader to feed a Delta table. The data engineer 
   maintaining the upstream system tells you they're going to start adding 
   a new optional field next week. You want zero pipeline disruption — no 
   manual intervention, no data loss. Which schema evolution mode do you 
   use, and what else do you need to configure? Why?

## Self Check Answers

1. `schemaLocation` and `checkpointLocation` are able to be the same path. When this happens, Auto Loader creates a `_schemas` subdirectory in the checkpoint directory. The schema is persisted in the `_schemas` folder after Auto Loader's initial schema inference on the first run. The checkpoint is written in `checkpointLocation` to track which files have been processed thus far. 
2. Deleting the checkpoint directory removes the Auto Loader's memory of which files have been processed. Thus the next run will process all files sitting in the directory at runtime, regardless of whether or not they are new or old files. If the only change is the deletion of the checkpoint, the next run will result in data duplication in the target. Checkpoints are the Auto Loader's mechanism for enforcing exactly-once-per-file ingestion to preserve data quality by preventing duplication. 
3. If the schema evolution mode is `addNewColumns`, the first run with a new schema will result in the stream stopping with an exception. Auto Loader reads the new schema and recognizes that it does not match the current schema, stops after the read, and updates the expected read-side schema. Assuming `writeStream` is using `mergeSchema=True`, the subsequent run will merge the new schema from the read to the existing Delta table's schema and perform the write seamlessly. 
4. File notification is the more approriate discovery mode for this situation. Assuming that processed files are neither deleted nor moved to an archive location, using List Directory mode would increase resource usage and costs every single run as the number of files being listed is growing by approximately 10,000 per day in an already sizeable directory. File notification scales with the rate of new files instead of the total count of files. Ingestion is event-driven, meaning that Auto Loader will only read and write the new files it has been notified about. File notification would only consume resources and accrue costs for 10,000 files a day rather than 50 million files and climbing.
5. Auto Loader would attempt to parse the CSV as JSON. By default, Auto Loader will fail the read. Depending on `mode` settings, the bad records are dropped or pushed into a corrupt-record column (`PERMISSIVE`, `DROPMALFORMED`, `FAILFAST`). Although it is unlikely, if the CSV contains data parseable as JSON, Auto Loader may sliently produce malformed rows. `.option("pathGlobFilter", "*.json")` can be used to define a filter for the directory that would ignore the CSV or other files.
6. In this scenario, using `addNewColumns` is the first option. This is the default mode for Auto Loader's schema evolution mode (`cloudFiles.schemaEvolutionMode`), but will require that the `spark.writeStream` includes `.option('mergeSchema', 'true')` as well. Once the new files are sent, the `spark.readStream` will stop with an exception on its first run so that Auto Loader can evolve the expected schema, then on retry (assuming retry logic is in place to prevent any need for intervention) `spark.readStream` succeeds, after which `spark.writeSchema.option('mergeSchema', 'true')...` can merge the new schema with the target Delta table's current schema and write the new data with the optional field. However, there is no way for Auto Loader to evolve the schema without stopping the stream. `addNewColumns` can limit or prevent intervention, but cannot prevent stream disruption in this scenario. The second option is `rescue` if the downstream workflows do not permit the schema change. `rescue` enables the new optional field to be written to and preserved in `_rescued_data` while maintaining the original fixed schema without requiring the stream to stop. 